In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA
import plotly.graph_objects as go

# ------------------------------------------------------------
# Paths and settings
# ------------------------------------------------------------

MAIN_WORKBOOK = Path("data") / "data.xlsx"
POSITIVE_SHEET = "Positive"
NEGATIVE_SHEET = "Negative"

OUTPUT_DIR = Path("outputs") / "figure_6_figure_s4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POINT_SIZE_3D = 1.5

# Toggle this to control whether positive samples are shown by class
# If False: all positives are shown as one group
# If True: positives are split into Lamp oil / White spirit / Diesel / Gasoline
SHOW_POSITIVE_CLASSES = True
POSITIVE_CLASS_ORDER = ["Lamp oil", "White spirit", "Diesel", "Gasoline"]
POSITIVE_CLASS_COLORS = {
    "Lamp oil": "royalblue",
    "White spirit": "seagreen",
    "Diesel": "darkorange",
    "Gasoline": "mediumpurple",
}

# ------------------------------------------------------------
# Root sets
# ------------------------------------------------------------

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12",
    "SH15", "SH16", "SH19", "SH20",
    "T3", "T6", "T9", "T12", "T15",
    "Te3", "Te6", "Te9", "Te12", "Te15",
}

GAS95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}

GAS98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}

# ------------------------------------------------------------
# Metadata and loading
# ------------------------------------------------------------

def build_metadata(df_pos: pd.DataFrame) -> pd.DataFrame:
    idx = df_pos.index.astype(str)
    roots = idx.to_series().str.split("-", n=1).str[0]

    simca_class = []
    for r in roots:
        if r.startswith("L"):
            simca_class.append("Lamp oil")
        elif r.startswith("W"):
            simca_class.append("White spirit")
        elif r in DIESEL_ROOTS:
            simca_class.append("Diesel")
        elif r in GAS95_ROOTS or r in GAS98_ROOTS:
            simca_class.append("Gasoline")
        elif r.startswith("B"):
            simca_class.append("Brandspiritus")
        else:
            raise ValueError(f"Unknown root code: {r}")

    meta = pd.DataFrame(
        {"root": roots.values, "simca_class": simca_class},
        index=df_pos.index,
    )
    return meta


def load_data():
    pos = pd.read_excel(MAIN_WORKBOOK, sheet_name=POSITIVE_SHEET)
    neg = pd.read_excel(MAIN_WORKBOOK, sheet_name=NEGATIVE_SHEET)

    pos = pos.set_index(pos.columns[0])
    neg = neg.set_index(neg.columns[0])

    pos.columns = pos.columns.astype(str)
    neg.columns = neg.columns.astype(str)

    cols_sorted = sorted(pos.columns, key=lambda x: float(x))
    pos = pos[cols_sorted]
    neg = neg[cols_sorted]

    meta_pos = build_metadata(pos)
    return pos, neg, meta_pos


# ------------------------------------------------------------
# Preprocessing
# ------------------------------------------------------------

def preproc_snv(df: pd.DataFrame) -> pd.DataFrame:
    X = df.to_numpy(dtype=float)
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, ddof=1, keepdims=True)
    std[std == 0.0] = 1.0
    Xn = (X - mean) / std
    return pd.DataFrame(Xn, index=df.index, columns=df.columns)


def preproc_sg(
    df: pd.DataFrame,
    polyorder: int,
    window_length: int,
    deriv: int,
) -> pd.DataFrame:
    X = df.to_numpy(dtype=float)
    Xf = savgol_filter(
        X,
        window_length=window_length,
        polyorder=polyorder,
        deriv=deriv,
        axis=1,
        mode="interp",
    )
    return pd.DataFrame(Xf, index=df.index, columns=df.columns)


def build_preprocess_configs():
    return [
        {"name": "snv", "kind": "snv"},
        {"name": "sg1_p2_w3", "kind": "sg", "poly": 2, "win": 3, "deriv": 1},
        {"name": "sg2_p2_w3", "kind": "sg", "poly": 2, "win": 3, "deriv": 2},
    ]


def apply_preprocess(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if cfg["kind"] == "snv":
        return preproc_snv(df)
    if cfg["kind"] == "sg":
        return preproc_sg(
            df,
            polyorder=cfg["poly"],
            window_length=cfg["win"],
            deriv=cfg["deriv"],
        )
    raise ValueError(f"Unknown preprocessing kind: {cfg['kind']}")


# ------------------------------------------------------------
# PCA helpers
# ------------------------------------------------------------

def fit_pca_and_scores(pos_proc: pd.DataFrame, neg_proc: pd.DataFrame):
    X_pos = pos_proc.to_numpy(dtype=float)
    X_neg = neg_proc.to_numpy(dtype=float)

    mean_ = X_pos.mean(axis=0)
    X_pos_centered = X_pos - mean_
    X_neg_centered = X_neg - mean_

    pca3 = PCA(n_components=3)
    pos_scores_3d = pca3.fit_transform(X_pos_centered)
    neg_scores_3d = pca3.transform(X_neg_centered)
    evr3 = pca3.explained_variance_ratio_ * 100.0

    return pos_scores_3d, neg_scores_3d, evr3


# ------------------------------------------------------------
# Plotting
# ------------------------------------------------------------

def make_3d_pca_html(
    pos_scores: np.ndarray,
    neg_scores: np.ndarray,
    evr: np.ndarray,
    pos_labels,
    neg_labels,
    pos_classes,
    preprocess_name: str,
):
    pos_hover = [f"{idx}<br>Positive" for idx in pos_labels]
    neg_hover = [f"{idx}<br>Negative" for idx in neg_labels]
    pos_classes = np.asarray(pos_classes)

    fig = go.Figure()

    if SHOW_POSITIVE_CLASSES:
        for cls in POSITIVE_CLASS_ORDER:
            mask = pos_classes == cls
            if not np.any(mask):
                continue
            cls_hover = [f"{idx}<br>{cls}" for idx in np.asarray(pos_labels)[mask]]
            fig.add_trace(
                go.Scatter3d(
                    x=pos_scores[mask, 0],
                    y=pos_scores[mask, 1],
                    z=pos_scores[mask, 2],
                    mode="markers",
                    name=cls,
                    text=cls_hover,
                    hovertemplate="%{text}<extra></extra>",
                    marker=dict(
                        size=POINT_SIZE_3D,
                        symbol="circle",
                        color=POSITIVE_CLASS_COLORS[cls],
                        opacity=0.85,
                        line=dict(color=POSITIVE_CLASS_COLORS[cls], width=0.6),
                    ),
                )
            )
    else:
        fig.add_trace(
            go.Scatter3d(
                x=pos_scores[:, 0],
                y=pos_scores[:, 1],
                z=pos_scores[:, 2],
                mode="markers",
                name="Positive",
                text=pos_hover,
                hovertemplate="%{text}<extra></extra>",
                marker=dict(
                    size=POINT_SIZE_3D,
                    symbol="circle",
                    color="royalblue",
                    opacity=0.85,
                    line=dict(color="royalblue", width=0.6),
                ),
            )
        )

    fig.add_trace(
        go.Scatter3d(
            x=neg_scores[:, 0],
            y=neg_scores[:, 1],
            z=neg_scores[:, 2],
            mode="markers",
            name="Negative",
            text=neg_hover,
            hovertemplate="%{text}<extra></extra>",
            marker=dict(
                size=POINT_SIZE_3D,
                symbol="circle-open",
                color="firebrick",
                opacity=1.0,
                line=dict(color="firebrick", width=0.9),
            ),
        )
    )

    fig.update_layout(
        title=preprocess_name,
        scene=dict(
            xaxis_title=f"PC1 ({evr[0]:.2f}%)",
            yaxis_title=f"PC2 ({evr[1]:.2f}%)",
            zaxis_title=f"PC3 ({evr[2]:.2f}%)",
        ),
        legend=dict(x=0.01, y=0.99),
        template="plotly_white",
        margin=dict(l=0, r=0, t=70, b=0),
    )

    out_html = OUTPUT_DIR / f"pca_3d_plot_{preprocess_name}.html"
    fig.write_html(str(out_html), include_plotlyjs=True)
    return out_html


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():
    pos_raw, neg_raw, meta_pos = load_data()

    # Brandspiritus is excluded from the hydrocarbon-class PCA.
    mask_non_brand = meta_pos["simca_class"] != "Brandspiritus"
    pos_raw = pos_raw.loc[mask_non_brand].copy()
    meta_pos = meta_pos.loc[pos_raw.index].copy()

    preprocess_configs = build_preprocess_configs()
    saved_3d_files = []

    for cfg in preprocess_configs:
        pos_proc = apply_preprocess(pos_raw, cfg)
        neg_proc = apply_preprocess(neg_raw, cfg)

        pos_scores, neg_scores, evr = fit_pca_and_scores(pos_proc, neg_proc)

        out_html = make_3d_pca_html(
            pos_scores=pos_scores,
            neg_scores=neg_scores,
            evr=evr,
            pos_labels=pos_proc.index.astype(str),
            neg_labels=neg_proc.index.astype(str),
            pos_classes=meta_pos.loc[pos_proc.index, "simca_class"].astype(str).values,
            preprocess_name=cfg["name"],
        )
        saved_3d_files.append(out_html)

    print(f"Saved {len(saved_3d_files)} interactive 3D PCA files:")
    for f in saved_3d_files:
        print(f)


if __name__ == "__main__":
    main()
